In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sparse
import itertools
import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, LinearSegmentedColormap
from datetime import date
import re
# from tqdm import tqdm
import time
# from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr, rankdata, describe, mode, fisher_exact
import scipy as sp
import scipy.stats as stats
# from scipy.spatial.distance import jaccard, euclidean, cosine
# import functools
# import operator
import gzip
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.metrics import pairwise_distances
import sys
import os
from functools import partial
from contextlib import closing
from multiprocessing import Pool
import re
import h5py
import tabix
import upsetplot
from wordcloud import WordCloud
from sklearn.decomposition import NMF, non_negative_factorization
from collections import Counter
from scipy.stats import hypergeom
from scipy.stats.mstats import gmean
from scipy.cluster import hierarchy
import squarify
try:
    import piplite
    await piplite.install(['ipywidgets'])
except ImportError:
    pass
import ipywidgets as widgets
import json
from subprocess import Popen, PIPE

today = date.today()


**Getting Ranks with Tabix**

In [4]:
def tabix_query(filename, query):
    # Use subprocess with tabix and return hits
    process = Popen(['tabix', '-f', filename, query], stdout=PIPE)
    
    hits = []
    for line in process.stdout:
        hits.append(np.char.decode(line.strip().split()))
    hits = np.array(hits)
    return hits

def load_roi(genomic_location, link_path):
    # determine roi from hdf5 scale file
    query_start_end = genomic_location.split(':')[1].split('-')
    query_start = int(query_start_end[0])
    query_end = int(query_start_end[1])

    possible_coords = tabix_query(link_path, genomic_location)

    start_dist = np.abs(possible_coords[:, 1].astype(int) - query_start)
    end_dist = np.abs(possible_coords[:, 2].astype(int) - query_end)

    query_region = possible_coords[np.argmin(start_dist + end_dist)]

    roi = int(query_region[3])
    roi_percents = query_region[4:].astype('float32')
    roi_coords = query_region[:3]
    return roi

In [44]:
scale = '2'
genomic_location = 'chrX:48778015-48802844'
link_path = 'https://resources.altius.org/~ctrader/public/genomic_narration/indexed/hg38.1kb.regions.CT20230427.sorted.rownum4.ranks5t38.bed.gz'


In [18]:
query_start_end = genomic_location.split(':')[1].split('-')
query_start = int(query_start_end[0])
query_end = int(query_start_end[1])

In [20]:
query_end

48802844

In [46]:
possible_coords = tabix_query(link_path, genomic_location)

In [47]:
possible_coords[:, 1].astype(int)

array([48778000, 48779000, 48780000, 48781000, 48782000, 48783000,
       48784000, 48785000, 48786000, 48787000, 48788000, 48789000,
       48790000, 48791000, 48792000, 48793000, 48794000, 48795000,
       48796000, 48797000, 48798000, 48799000, 48800000, 48801000,
       48802000])

In [22]:
start_dist = np.abs(possible_coords[:, 1].astype(int) - query_start)
start_dist

array([ 1015,    15,   985,  1985,  2985,  3985,  4985,  5985,  6985,
        7985,  8985,  9985, 10985, 11985, 12985, 13985, 14985, 15985,
       16985, 17985, 18985, 19985, 20985, 21985, 22985, 23985])

In [23]:
end_dist = np.abs(possible_coords[:, 2].astype(int) - query_end)
end_dist

array([23844, 22844, 21844, 20844, 19844, 18844, 17844, 16844, 15844,
       14844, 13844, 12844, 11844, 10844,  9844,  8844,  7844,  6844,
        5844,  4844,  3844,  2844,  1844,   844,   156,  1156])

In [26]:
np.argmin(start_dist + end_dist)

2

In [28]:
query_region = possible_coords[np.argmin(start_dist + end_dist)]
query_region

array(['chrX', '48779000', '48781000', '2923791', '0.5828351',
       '0.7806531', '0.84861785', '0.6559117', '0.0', '0.0', '0.77121246',
       '0.9252815', '0.6827333', '0.0', '0.0', '0.0', '0.0', '0.0',
       '0.8050867', '0.9013164', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0',
       '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0',
       '0.0', '0.8703793', '0.54831266'], dtype='<U10')

In [30]:
roi = int(query_region[3])
roi

2923791

In [39]:
roi_percents = query_region[4:].astype('float32')
roi_percents

array([0.5828351 , 0.7806531 , 0.84861785, 0.6559117 , 0.        ,
       0.        , 0.77121246, 0.9252815 , 0.6827333 , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.8050867 ,
       0.9013164 , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.8703793 , 0.54831266], dtype=float32)

In [40]:
roi_coords = query_region[:3]
roi_coords

array(['chrX', '48779000', '48781000'], dtype='<U10')